# 03 — Model Training & Evaluation
### Landslide Risk Monitoring | SIH 26001 | NER

This notebook covers:
- Feature engineering
- Model training (XGBoost / Random Forest)
- Evaluation metrics
- Feature importance
- Sample predictions with risk scoring

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay
from sklearn.model_selection import train_test_split

REPO_ROOT = Path().resolve().parents[1]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

sns.set_theme(style='darkgrid')
plt.rcParams['figure.dpi'] = 100
print('✓ Imports OK | Repo root:', REPO_ROOT)

## 1. Load Cleaned Data

In [ ]:
from ml.preprocessing.cleaning import load_data, clean_data
from ml.preprocessing.feature_engineering import engineer_features, get_feature_columns
from ml.datasets.generate_sample import generate_sample_dataset

SAMPLE_PATH = REPO_ROOT / 'ml' / 'datasets' / 'sample' / 'landslide_sample.csv'

if not SAMPLE_PATH.exists():
    generate_sample_dataset(n_samples=1000)

df_raw = load_data(SAMPLE_PATH)
df_clean = clean_data(df_raw, verbose=False)
print(f'Cleaned dataset: {df_clean.shape}')

## 2. Feature Engineering

In [ ]:
df_feat = engineer_features(df_clean, verbose=True)
feature_cols = get_feature_columns()

print(f'\nEngineered features ({len(feature_cols)}):')
for col in feature_cols:
    print(f'  {col}')

df_feat[feature_cols].describe().round(3)

## 3. Feature Correlation with Target

In [ ]:
correlations = df_feat[feature_cols + ['landslide_occurred']].corr()['landslide_occurred'].drop('landslide_occurred').sort_values()

plt.figure(figsize=(10, 5))
colors = ['#e74c3c' if v > 0 else '#3498db' for v in correlations.values]
bars = plt.barh(correlations.index, correlations.values, color=colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel('Correlation with landslide_occurred')
plt.title('Engineered Feature Correlations with Target', fontweight='bold')
plt.tight_layout()
plt.show()

print('\nCorrelations:')
print(correlations.round(3))

## 4. Train Model

In [ ]:
from ml.models.landslide_model import train_model, save_model

model, metrics = train_model(df_feat)
print('\n=== Training Complete ===')
print(f'Algorithm : {type(model.named_steps["classifier"]).__name__}')
print(f'Test size : {metrics["test_samples"]} samples')

## 5. Evaluation Metrics

In [ ]:
print('=== MODEL EVALUATION RESULTS ===')
metric_labels = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
for m in metric_labels:
    val = metrics[m]
    bar = '█' * int(val * 30)
    print(f'  {m:<12}: {val:.4f}  {bar}')

print(f'\n  Class distribution (test set):')
print(f'    Positive (landslide=1): {metrics["class_distribution"]["positive"]}')
print(f'    Negative (landslide=0): {metrics["class_distribution"]["negative"]}')

## 6. Confusion Matrix

In [ ]:
# Recreate test split for plotting
X = df_feat[feature_cols].values.astype('float32')
y = df_feat['landslide_occurred'].astype(int).values
_, X_test, _, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix
ConfusionMatrixDisplay.from_estimator(
    model, X_test, y_test,
    display_labels=['No Landslide', 'Landslide'],
    cmap='Blues', ax=axes[0]
)
axes[0].set_title('Confusion Matrix', fontweight='bold')

# ROC curve
RocCurveDisplay.from_estimator(model, X_test, y_test, ax=axes[1], color='#e74c3c')
axes[1].plot([0,1],[0,1],'--', color='grey', alpha=0.6, label='Random')
axes[1].set_title(f'ROC Curve (AUC={metrics["roc_auc"]:.3f})', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

## 7. Feature Importance

In [ ]:
if 'feature_importance' in metrics:
    fi = pd.Series(metrics['feature_importance']).sort_values()
    
    plt.figure(figsize=(10, 6))
    colors = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(fi)))
    bars = plt.barh(fi.index, fi.values, color=colors)
    plt.xlabel('Feature Importance Score')
    plt.title('Feature Importance', fontweight='bold', fontsize=13)
    for bar, val in zip(bars, fi.values):
        plt.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
                 f'{val:.4f}', va='center', fontsize=9)
    plt.tight_layout()
    plt.show()
else:
    print('Feature importance not available for this model type.')

## 8. Save Model

In [ ]:
MODEL_DIR = REPO_ROOT / 'ml' / 'models' / 'saved'
model_path = save_model(model, metrics, model_dir=MODEL_DIR)
print(f'✓ Model saved: {model_path}')

## 9. Sample Predictions with Risk Scoring

In [ ]:
from ml.prediction.predict import predict_risk, RISK_THRESHOLDS

test_cases = [
    {'location_id': 201, 'rainfall_mm': 350, 'soil_moisture': 92, 'slope_degree': 52,
     'elevation_m': 1800, 'terrain_roughness': 0.88, 'historical_landslide_count': 15},
    {'location_id': 202, 'rainfall_mm': 20, 'soil_moisture': 25, 'slope_degree': 8,
     'elevation_m': 120, 'terrain_roughness': 0.10, 'historical_landslide_count': 0},
    {'location_id': 203, 'rainfall_mm': 120, 'soil_moisture': 55, 'slope_degree': 28,
     'elevation_m': 700, 'terrain_roughness': 0.50, 'historical_landslide_count': 3},
    {'location_id': 204, 'rainfall_mm': 200, 'soil_moisture': 78, 'slope_degree': 40,
     'elevation_m': 1200, 'terrain_roughness': 0.72, 'historical_landslide_count': 9},
]

print('=== SAMPLE PREDICTIONS WITH RISK SCORING ===')
print(f'{"Loc ID":>6} | {"Score":>5} | {"Level":<8} | {"Confidence":>10} | Input Summary')
print('-' * 80)

results = []
level_emoji = {'LOW': '🟢', 'MEDIUM': '🟡', 'HIGH': '🔴', 'CRITICAL': '🔴'}

for case in test_cases:
    r = predict_risk(case, model=model)
    results.append(r)
    emoji = level_emoji.get(r['risk_level'], '?')
    print(f"{r['location_id']:>6} | {r['risk_score']:>5} | "
          f"{emoji} {r['risk_level']:<6} | {r['confidence']:>10.4f} | "
          f"rain={case['rainfall_mm']}mm slope={case['slope_degree']}°")

print()
import json
print('Full JSON output for location 201:')
print(json.dumps(results[0], indent=2))

## 10. Risk Score Distribution on Test Set

In [ ]:
from ml.prediction.predict import probability_to_risk_score, risk_score_to_level

y_prob_test = model.predict_proba(X_test)[:, 1]
risk_scores  = [probability_to_risk_score(p) for p in y_prob_test]
risk_levels  = [risk_score_to_level(s) for s in risk_scores]

level_order = ['LOW', 'MEDIUM', 'HIGH', 'CRITICAL']
level_colors = {'LOW': '#2ecc71', 'MEDIUM': '#f39c12', 'HIGH': '#e67e22', 'CRITICAL': '#e74c3c'}
level_counts = pd.Series(risk_levels).value_counts().reindex(level_order, fill_value=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Risk level distribution
axes[0].bar(level_counts.index, level_counts.values,
            color=[level_colors[l] for l in level_counts.index])
axes[0].set_title('Risk Level Distribution (Test Set)', fontweight='bold')
axes[0].set_ylabel('Count')
for i, (idx, val) in enumerate(level_counts.items()):
    axes[0].text(i, val + 0.5, str(val), ha='center', fontweight='bold')

# Risk score histogram
axes[1].hist(risk_scores, bins=20, color='#3498db', edgecolor='white', alpha=0.85)
for level, (lo, hi) in RISK_THRESHOLDS.items():
    axes[1].axvspan(lo, hi, alpha=0.08, color=list(level_colors.values())[list(RISK_THRESHOLDS.keys()).index(level)])
    axes[1].text((lo+hi)/2, axes[1].get_ylim()[1]*0.9, level, ha='center', fontsize=7, color='black')
axes[1].set_title('Risk Score Distribution (Test Set)', fontweight='bold')
axes[1].set_xlabel('Risk Score (0–100)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

## Summary

In [ ]:
print('=== TRAINING PIPELINE SUMMARY ===')
print(f'Algorithm  : {type(model.named_steps["classifier"]).__name__}')
print(f'Accuracy   : {metrics["accuracy"]:.4f}')
print(f'Precision  : {metrics["precision"]:.4f}')
print(f'Recall     : {metrics["recall"]:.4f}')
print(f'F1 Score   : {metrics["f1"]:.4f}')
print(f'ROC-AUC    : {metrics["roc_auc"]:.4f}')
print(f'Model saved: {model_path}')
print()
print('To use in production backend:')
print('  from ml.prediction.predict import predict_risk, load_model')
print('  model = load_model()')
print('  result = predict_risk(input_dict, model=model)')